# Experiment 1: Sugar GRN Activation - Frequency Sweep

**Purpose**
- Reproduce Shiu et al. (2024) Figure 1c-d
- Test neural response across 20 activation frequencies (10-200 Hz)
- Validate against original paper results
- Generate Top 200 responsive neurons for Exp2/3

**Validation & Controls**
- Correlation with Shiu et al. (2024) results (r > 0.85 expected)
- ID version conversion (v630 → v783) with verification
- Shuffled connectivity control (proves model depends on true connectome)
- Multi-motor neuron analysis (MN6, 8, 9, 11 responses)

**Configuration**
- Neurons: 21 Sugar GRNs (left hemisphere, v783)
- Frequencies: 10, 20, ..., 200 Hz (20 frequencies)
- Trials: 10 per frequency
- Batches: 4 batches × 5 frequencies (for memory management)
- Workers: 3 (4-core machine)

**Expected Runtime**
- Per batch: ~15-20 minutes
- Total: ~1.5 hours (can run overnight)

**Outputs**
- `results/exp1_sugar_activation/exp1_complete_results.pkl`
- `results/exp1_sugar_activation/top_200_neurons.npy` (for Exp2/3)
- `results/exp1_sugar_activation/figures/` (6 plots)
  - Correlation with original paper
  - Network-wide response heatmap
  - Proboscis motor neurons (MN6, 8, 9, 11)
  - Summary statistics
  - Shuffled connectivity control
---

## 1. Environment Setup

In [ ]:
import numpy as np
import pandas as pd
import sys
from pathlib import Path

from time import time
import pickle

import matplotlib.pyplot as plt
from scipy.stats import pearsonr
import seaborn as sns

from brian2 import Hz, ms, start_scope, volt

from joblib import Parallel, delayed
import gc

print("✅ Standard libraries imported")

In [ ]:
# FlyWire tools
import pyarrow.parquet as pq
from caveclient import CAVEclient

print("Initializing CAVEclient...")
client = CAVEclient('flywire_fafb_public')
print("✅ Connected to FlyWire")

In [ ]:
# FlyLIF modules
from flylif.core.parameters import DEFAULT_PARAMS
from flylif.core.data_loader import load_simulation_data
from flylif.core.network import build_network
from flylif.core.simulation import run_simulation
from flylif.utils.visualization import plot_correlation, plot_response_heatmap, plot_frequency_response_curve, plot_summary_statistics
from flylif.utils.cave_utils import convert_neuron_list_cached
from flylif.utils.checkpoint import CheckpointManager
from flylif.utils.memory_utils import print_memory, check_memory_safe, memory_cleanup, MemoryMonitor

print("✅ FlyLIF modules imported")

In [ ]:
# Auto-reload for development
%load_ext autoreload
%autoreload 2
print("✅ Auto-reload enabled")

## 2. Configuration

In [ ]:
# Base directory
BASE_DIR = Path.home() / 'mylibrary'/'connectome'/'lifmodel'

CONFIG = {
    'base_dir': BASE_DIR,
    'data_dir': BASE_DIR / 'data_783',
    'connections_file': 'proofread_connections_783.feather',
    'root_ids_file': 'proofread_root_ids_783.npy',
    'classification': 'classification.csv',
}

print("Configuration:")
print(f"  Data dir: {CONFIG['data_dir']}")
print(f"  Connections: {CONFIG['connections_file']}")
print(f"  Root IDs: {CONFIG['root_ids_file']}")

## 3. Define Neuron IDs

### 3.1 Original v630 IDs

In [ ]:
print("=" * 70)
print("Defining Original Paper Neuron IDs (v630)")
print("=" * 70)

# Original paper neurons (from Shiu et al., FlyWire v630)
NEU_SUGAR_v630 = [
    720575940624963786, 720575940630233916, 720575940637568838, 720575940638202345, 720575940617000768,
    720575940630797113, 720575940632889389, 720575940621754367, 720575940621502051, 720575940640649691,
    720575940639332736, 720575940616885538, 720575940639198653, 720575940620900446, 720575940617937543,
    720575940632425919, 720575940633143833, 720575940612670570, 720575940628853239, 720575940629176663,
    720575940611875570
]

NEU_MN9_v630 = [720575940660219265]  # MN9 right (contralateral)
NEU_MN9_LEFT_v630 = [720575940645521262]  # MN9 left (ipsilateral)

print(f"\nOriginal neuron counts:")
print(f"  Sugar GRNs: {len(NEU_SUGAR_v630)}")
print(f"  MN9 Right: {len(NEU_MN9_v630)}")
print(f"  MN9 Left: {len(NEU_MN9_LEFT_v630)}")

### 3.2 Convert to v783

In [ ]:
print("\n" + "=" * 70)
print("Converting Neuron IDs (v630 → v783)")
print("=" * 70)
print("⚠️  This uses FlyWire API, will be cached to avoid repeated calls\n")

# Cache directory
cache_dir = BASE_DIR / 'flylif' / 'cache' / 'id_conversions'
cache_dir.mkdir(parents=True, exist_ok=True)

t0 = time()

# Convert Sugar GRNs
NEU_SUGAR_LEFT = convert_neuron_list_cached(
    old_list=NEU_SUGAR_v630,
    cache_file=cache_dir / 'sugar_grns_v630_to_v783.pkl',
    new_ver=783,
    force_update=False,
    verbose=True
)

# Convert MN9
NEU_MN9_RIGHT = convert_neuron_list_cached(
    old_list=NEU_MN9_v630,
    cache_file=cache_dir / 'mn9_v630_to_v783.pkl',
    new_ver=783,
    verbose=True
)

NEU_MN9_LEFT = convert_neuron_list_cached(
    old_list=NEU_MN9_LEFT_v630,
    cache_file=cache_dir / 'mn9_left_v630_to_v783.pkl',
    new_ver=783,
    verbose=True
)

conversion_time = time() - t0

print(f"\n✅ Conversion complete in {conversion_time:.1f}s")
print(f"\n   Sugar GRNs (v783): {len(NEU_SUGAR_LEFT)}")
print(f"   MN9 Right: {NEU_MN9_RIGHT[0]}")
print(f"   MN9 Left: {NEU_MN9_LEFT[0]}")

### 3.3 Save converted IDs

In [ ]:
# Save converted neuron lists for future use
neuron_ids_v783 = {
    'NEU_SUGAR_LEFT': NEU_SUGAR_LEFT,
    'NEU_MN9_RIGHT': NEU_MN9_RIGHT,
    'NEU_MN9_LEFT': NEU_MN9_LEFT,
    'conversion_time': conversion_time,
    'source_version': 630,
    'target_version': 783,
}

backup_file = cache_dir / 'neuron_ids_v783_exp1.pkl'
with open(backup_file, 'wb') as f:
    pickle.dump(neuron_ids_v783, f)

print(f"✅ Neuron IDs backup: {backup_file}")

## 4. Load Data

In [ ]:
print("\n" + "=" * 70)
print("Loading Optimized Connectivity Data")
print("=" * 70)

print_memory("Before loading: ")

t0 = time()
DATA = load_simulation_data(CONFIG)

print(f"\n✅ Loaded in {time()-t0:.1f}s")
print(f"   Data size: {DATA['df_conn'].memory_usage(deep=True).sum()/1e6:.0f} MB")
print(f"   Neurons: {DATA['n_neurons']:,}")
print(f"   Connections: {len(DATA['df_conn']):,}")

print_memory("After loading: ")

## 5. Batch Configuration

In [ ]:
# Define 4 batches, 5 frequencies each
BATCH_CONFIG = {
    'batch1': [10, 20, 30, 40, 50],
    'batch2': [60, 70, 80, 90, 100],
    'batch3': [110, 120, 130, 140, 150],
    'batch4': [160, 170, 180, 190, 200],
}

print("Batch configuration:")
for name, freqs in BATCH_CONFIG.items():
    print(f"  {name}: {freqs}")

print(f"\nTotal frequencies: {sum(len(f) for f in BATCH_CONFIG.values())}")

## 6. Worker Function & Batch Runner

In [ ]:
# Worker function with memory optimization
def run_freq_worker_clean(freq, neu_exc, data, params, n_trials):
    """
    Worker with explicit memory cleanup and Brian2 isolation.
    
    Optimizations:
    - start_scope() for clean state
    - Minimal return data
    - Explicit cleanup with gc.collect()
    """
    start_scope()  # Clear Brian2 global state
    
    from flylif.core.network import build_network
    from flylif.core.simulation import run_simulation
    
    # Build network
    columns = data['columns']
    net_components = build_network(
        data=data,
        pre_col=columns['pre_col'],
        post_col=columns['post_col'],
        weight_col=columns['weight_col'],
        nt_prob_cols=columns.get('nt_prob_cols', {}),
        params=params,
        syn_threshold=5,
        verbose=False
    )
    
    # Run simulation
    result = run_simulation(
        net_components=net_components,
        neu_exc=neu_exc,
        params={'r_poi': freq * Hz},
        n_trials=n_trials,
        verbose=False
    )
    
    # Extract only essential data
    result_clean = {
        'n_active': result['n_active'],
        'n_spikes': result['n_spikes'],
        'df': result['df'].copy(),
    }
    
    # Explicit cleanup
    del net_components
    del result
    gc.collect()
    
    return (freq, result_clean)

In [ ]:
# Batch runner with checkpoint and memory monitoring
def run_batch(freq_list, batch_name, n_workers=3, n_trials=10, 
              checkpoint_dir='./checkpoints/exp1'):
    """
    Run batch with checkpoint and memory monitoring.
    
    Parameters
    ----------
    freq_list : list
        List of frequencies to test
    batch_name : str
        Name of the batch (for logging)
    n_workers : int
        Number of parallel workers
    n_trials : int
        Trials per frequency
    checkpoint_dir : str or Path
        Checkpoint directory
    
    Returns
    -------
    dict : {freq: result}
    """
    print("\n" + "=" * 70)
    print(f"Batch: {batch_name}")
    print("=" * 70)
    
    # Initialize checkpoint
    ckpt = CheckpointManager(checkpoint_dir)
    
    # Check remaining tasks
    remaining = ckpt.get_remaining(freq_list)
    
    if not remaining:
        print(f"✅ All frequencies already completed, loading from checkpoint...")
        return ckpt.load_all_completed()
    
    print(f"\nConfiguration:")
    print(f"  Frequencies: {remaining}")
    print(f"  Trials/freq: {n_trials}")
    print(f"  Workers: {n_workers}")
    print(f"  Checkpoint: {checkpoint_dir}")
    
    # Memory check
    is_safe, msg = check_memory_safe(threshold=80.0, swap_threshold=0.5)
    print(f"\n  Memory status: {msg}")
    
    if not is_safe:
        print(f"  ⚠️  Memory not safe, recommend cleanup first")
        memory_cleanup()
        print_memory("    After cleanup: ")
    
    # Prepare tasks
    tasks = [(freq, NEU_SUGAR_LEFT, DATA, DEFAULT_PARAMS, n_trials) 
             for freq in remaining]
    
    print(f"\n  Running {len(tasks)} tasks...")
    
    with MemoryMonitor(label=batch_name):
        t0 = time()
        
        results = Parallel(n_jobs=n_workers, verbose=5, max_nbytes='100M')(
            delayed(run_freq_worker_clean)(*task) for task in tasks
        )
        
        batch_time = time() - t0
    
    # Save to checkpoint
    for freq, result in results:
        ckpt.save(freq, result)
    
    print(f"\n  ✅ Batch complete in {batch_time/60:.2f} min")
    print_memory("  Final memory: ")
    
    # Cleanup
    del results
    gc.collect()
    
    return ckpt.load_all_completed()

## 7. Execute Batches

**Instructions**: Run each batch cell separately. Can restart kernel between batches if needed.

### 7.1 Batch 1 (10-50 Hz)

In [ ]:
results_batch1 = run_batch(
    freq_list=BATCH_CONFIG['batch1'],
    batch_name='Batch 1',
    n_workers=4,
    n_trials=10
)

print("\n" + "=" * 70)
print("✅ Batch 1 complete!")
print("=" * 70)
print("\nNext: Run Cell 7.2 (Batch 2)")
print("Optional: Kernel → Restart Kernel (if memory high)")

### 7.2 Batch 2 (60-100 Hz)

In [ ]:
# ⚠️  If restarted kernel, re-run Cells 1-6 first!

results_batch2 = run_batch(
    freq_list=BATCH_CONFIG['batch2'],
    batch_name='Batch 2',
    n_workers=4,
    n_trials=10
)

print("\n✅ Batch 2 complete!")

### 7.3 Batch 3 (110-150 Hz)

In [ ]:
# ⚠️  If restarted kernel, re-run Cells 1-6 first!

results_batch3 = run_batch(
    freq_list=BATCH_CONFIG['batch3'],
    batch_name='Batch 3',
    n_workers=4,
    n_trials=10
)

print("\n✅ Batch 3 complete!")

### 7.4 Batch 4 (160-200 Hz)

In [ ]:
# ⚠️  If restarted kernel, re-run Cells 1-6 first!

results_batch4 = run_batch(
    freq_list=BATCH_CONFIG['batch4'],
    batch_name='Batch 4 (Final)',
    n_workers=4,
    n_trials=10
)

print("\n✅ Batch 4 complete!")
print("\nNext: Run Cell 8 (Merge all batches)")

## 8. Merge All Batches

In [ ]:
print("\n" + "=" * 70)
print("Merging All Batches")
print("=" * 70)

# Load from checkpoint (in case kernel was restarted)
ckpt = CheckpointManager('./checkpoints/exp1')
all_results = ckpt.load_all_completed()

print(f"\n✅ Loaded {len(all_results)} frequencies from checkpoint")

# Organize results
results_dict = all_results

# Create summary
summary = []
for freq in sorted(all_results.keys()):
    res = all_results[freq]
    summary.append({
        'Frequency': freq,
        'Active Neurons': res['n_active'],
        'Total Spikes': res['n_spikes'],
    })

df_summary = pd.DataFrame(summary)

print(f"\nSummary:")
print(df_summary.to_string(index=False))

## 9. Calculate Firing Rates

In [ ]:
print("\n" + "=" * 70)
print("Calculating Firing Rates")
print("=" * 70)

# Extract firing rates for all neurons
all_firing_rates = {}
duration_s = float(DEFAULT_PARAMS['t_run'] / ms) / 1000

for freq, res in results_dict.items():
    df = res['df']
    if len(df) > 0:
        spike_counts = df.groupby('flywire_id').size()
        firing_rates = spike_counts / (10 * duration_s)  # 10 trials
        all_firing_rates[freq] = firing_rates.to_dict()
    else:
        all_firing_rates[freq] = {}

print(f"✅ Processed {len(all_firing_rates)} frequencies")

# Check maximum firing rate across all frequencies
print(f"\nFiring rate statistics:")
for freq in sorted(all_firing_rates.keys()):
    rates = list(all_firing_rates[freq].values())
    if rates:
        print(f"  {freq:3d} Hz: max={max(rates):6.1f} Hz, active neurons={len(rates)}")
    else:
        print(f"  {freq:3d} Hz: No activity")

## 10. Validation: Comparison with Original Paper

This section compares our results with Shiu et al. (2024) at 100Hz.

### 10.1 Load original data

In [ ]:
print("\n" + "=" * 70)
print("Loading Original Paper Results (v630, 100Hz)")
print("=" * 70)

# Path to original data
orig_data_path = BASE_DIR / 'Drosophila_brain_model/results/example/sugarR_100Hz.parquet'

if not orig_data_path.exists():
    print(f"\n⚠️  Original data not found: {orig_data_path}")
    print(f"   Skipping validation (will still save results)")
    has_original = False
else:
    print(f"\nLoading original spike data...")
    
    pf = pq.ParquetFile(orig_data_path)
    df_orig_v630 = pd.DataFrame({
        'flywire_id': pf.read(['flywire_id']).column('flywire_id').to_pylist(),
    })
    
    # Calculate original firing rates (v630 IDs)
    spike_counts_v630 = df_orig_v630.groupby('flywire_id').size()
    rate_orig_v630 = spike_counts_v630 / 30.0  # 30 trials in original
    rate_orig_v630.name = 'Original_v630'
    
    # Our results (v783 IDs)
    rate_ours_v783 = pd.Series(all_firing_rates[100], name='Ours_v783')
    
    print(f"   ✅ Loaded")
    print(f"      Original (v630): {len(rate_orig_v630)} active neurons")
    print(f"      Ours (v783):     {(rate_ours_v783>0).sum()} active neurons")
    
    has_original = True

### 10.2 Identify ID discrepancies

In [ ]:
if has_original:
    print("\n" + "=" * 70)
    print("Analyzing ID Discrepancies")
    print("=" * 70)
    
    # Find discrepancies
    ids_only_orig = set(rate_orig_v630.index) - set(rate_ours_v783.index)
    ids_only_ours = set(rate_ours_v783[rate_ours_v783>0].index) - set(rate_orig_v630.index)
    
    print(f"\n  Only in original (v630): {len(ids_only_orig)} neurons")
    print(f"  Only in ours (v783):     {len(ids_only_ours)} neurons")
    
    # Common neurons with large firing rate differences
    common_ids = set(rate_orig_v630.index) & set(rate_ours_v783.index)
    df_common = pd.DataFrame({
        'orig': rate_orig_v630[list(common_ids)],
        'ours': rate_ours_v783[list(common_ids)]
    }).fillna(0)
    
    # Active in both
    active_both = (df_common['orig'] > 0) & (df_common['ours'] > 0)
    df_active_common = df_common[active_both].copy()
    df_active_common['abs_diff'] = np.abs(df_active_common['orig'] - df_active_common['ours'])
    
    # Large difference (>20 Hz or >50% relative)
    large_diff_mask = (df_active_common['abs_diff'] > 20) | \
                      (df_active_common['abs_diff'] / df_active_common['orig'] > 0.5)
    ids_large_diff = df_active_common[large_diff_mask].index.tolist()
    
    print(f"  Large differences: {len(ids_large_diff)} neurons (>20Hz or >50%)")
    
    # Combine all discrepancy types
    ids_need_conversion = list(ids_only_orig) + ids_large_diff
    
    print(f"\n[Summary]")
    print(f"  Total IDs to convert: {len(ids_need_conversion)}")
    print(f"  Breakdown:")
    print(f"    - Missing from our data: {len(ids_only_orig)}")
    print(f"    - Large firing rate diff: {len(ids_large_diff)}")
    
    # Show examples
    if ids_only_orig:
        print(f"\n  Examples (missing IDs): {list(ids_only_orig)[:5]}")
    
    if ids_large_diff:
        top_diff = df_active_common.nlargest(5, 'abs_diff')
        print(f"\n  Top 5 firing rate discrepancies:")
        for idx, row in top_diff.iterrows():
            print(f"    {idx}: orig={row['orig']:.1f}, ours={row['ours']:.1f}, diff={row['abs_diff']:.1f}")

### 10.3 Convert discrepancy IDs

In [ ]:
if has_original and 'ids_need_conversion' in dir() and len(ids_need_conversion) > 0:
    print("\n" + "=" * 70)
    print(f"Converting {len(ids_need_conversion)} Discrepancy IDs (v630 → v783)")
    print("=" * 70)
    
    cache_file_disc = cache_dir / 'discrepancy_ids_v630_to_v783.pkl'
    
    t0 = time()
    
    ids_converted_v783 = convert_neuron_list_cached(
        old_list=ids_need_conversion,
        cache_file=cache_file_disc,
        new_ver=783,
        verbose=True
    )
    
    conversion_time = time() - t0
    
    print(f"\n✅ Conversion complete in {conversion_time:.1f}s")
    
    # Create mapping (v630 → v783)
    id_mapping = {old: new for old, new in zip(ids_need_conversion, ids_converted_v783)}
    
    # Check how many actually changed
    n_changed = sum(1 for old, new in id_mapping.items() if old != new)
    n_unchanged = len(id_mapping) - n_changed
    
    print(f"\n  ID changes: {n_changed} changed, {n_unchanged} unchanged")
    
    if n_changed > 0:
        print(f"\n  Changed examples (first 5):")
        count = 0
        for old, new in id_mapping.items():
            if old != new and count < 5:
                print(f"    {old} → {new}")
                count += 1
else:
    print("\n⚠️  No discrepancies to convert (or no original data)")
    id_mapping = {}

### 10.4 Re-compare with unified v783 IDs

In [ ]:
if has_original and 'id_mapping' in dir():
    print("\n" + "=" * 70)
    print("Re-comparison with v783 Unified IDs")
    print("=" * 70)
    
    # Reload full original spike data
    print("\nRe-loading full original spike data...")
    df_orig_full = pd.DataFrame({
        't': pf.read(['t']).column('t').to_pylist(),
        'trial': pf.read(['trial']).column('trial').to_pylist(),
        'flywire_id': pf.read(['flywire_id']).column('flywire_id').to_pylist(),
    })
    
    # Apply ID mapping (only remap discrepancy IDs)
    if len(id_mapping) > 0:
        print(f"  Applying ID mapping ({len(id_mapping)} IDs)...")
        df_orig_full['flywire_id'] = df_orig_full['flywire_id'].map(
            lambda x: id_mapping.get(x, x)
        )
        n_remapped = df_orig_full['flywire_id'].isin(id_mapping.values()).sum()
        print(f"  Remapped {n_remapped:,} spikes")
    
    # Recalculate firing rates (now all v783)
    spike_counts_orig = df_orig_full.groupby('flywire_id').size()
    rate_orig = spike_counts_orig / 30.0
    rate_orig.name = 'Original_v783'
    
    # Our results (already v783)
    rate_ours = pd.Series(all_firing_rates[100], name='Ours_v783')
    
    # === Correlation comparison ===
    print(f"\n[Correlation Analysis]")
    
    # Before conversion (for reference)
    df_compare_old = pd.concat([rate_orig_v630, rate_ours], axis=1).fillna(0)
    r_old, _ = pearsonr(df_compare_old.iloc[:,0], df_compare_old.iloc[:,1])
    
    # After conversion
    df_compare = pd.concat([rate_orig, rate_ours], axis=1).fillna(0)
    r, p = pearsonr(df_compare['Original_v783'], df_compare['Ours_v783'])
    
    print(f"\n  Before ID conversion: r = {r_old:.4f} (v630 vs v783)")
    print(f"  After ID conversion:  r = {r:.4f} (v783 vs v783)")
    print(f"  Improvement:          Δr = {r - r_old:+.4f}")
    
    if r > r_old + 0.01:
        print(f"  ✅ ID conversion improved correlation!")
    else:
        print(f"  ✓ ID version has minimal impact")
    
    # Active neuron counts
    print(f"\n  Active neurons:")
    print(f"    Original: {(df_compare['Original_v783']>0).sum()}")
    print(f"    Ours:     {(df_compare['Ours_v783']>0).sum()}")
    
    # Top neurons overlap
    top_n = 50
    top_orig = df_compare['Original_v783'].nlargest(top_n).index
    top_ours = df_compare['Ours_v783'].nlargest(top_n).index
    overlap = len(set(top_orig) & set(top_ours))
    
    print(f"\n  Top {top_n} overlap: {overlap}/{top_n} ({100*overlap/top_n:.0f}%)")
    
    # Final assessment
    print(f"\n[Validation Assessment]")
    if r > 0.90:
        print(f"  ✅ Excellent correlation (r = {r:.4f} > 0.90)")
        print(f"  ✅ Core methodology validated")
    elif r > 0.85:
        print(f"  ✅ Very good correlation (r = {r:.4f} > 0.85)")
        print(f"  ✓ Acceptable given trial count difference (10 vs 30)")
    elif r > 0.80:
        print(f"  ✓ Good correlation (r = {r:.4f} > 0.80)")
        print(f"  ⚠️  Consider increasing trials to 30 for better match")
    else:
        print(f"  ⚠️  Moderate correlation (r = {r:.4f})")
        print(f"  ⚠️  May need investigation")
    
    print("\n" + "=" * 70)
else:
    print("\n⚠️  Skipping validation (no original data)")

### 10.5 Shuffled connectivity control

In [ ]:
# ==================== Section 10.5: Shuffled Connectivity Control（修正版） ====================

print("\n" + "=" * 70)
print("Shuffled Connectivity Control (Corrected)")
print("=" * 70)

# Configuration
N_SIMULATIONS = 100
SHUFFLE_CHECKPOINT_DIR = './checkpoints/exp1_shuffled'
SHUFFLE_RESULTS_FILE = Path('./results/exp1_full/shuffled_connectivity_results.pkl')

In [ ]:
# Worker function
def shuffled_control_worker(sim_idx, shuffle, data, params, 
                            neu_exc, target_id, task_key=None):
    """
    Shuffled connectivity control worker (corrected).
    
    Shuffles connection topology by randomizing post-synaptic targets
    while maintaining global connectivity statistics.
    
    Parameters
    ----------
    sim_idx : int
        Simulation index (0-99)
    shuffle : bool
        If True, shuffle post-synaptic targets in df_conn
    data : dict
        Original DATA from load_simulation_data()
    params : dict
        DEFAULT_PARAMS
    neu_exc : list
        Neurons to activate (Sugar GRNs)
    target_id : int
        Target neuron to monitor (MN9)
    task_key : str
        Checkpoint key
    
    Returns
    -------
    tuple : (sim_idx, shuffle, mn9_activated, task_key)
    """
    from brian2 import start_scope, Hz
    start_scope()
    
    from flylif.core.network import build_network
    from flylif.core.simulation import run_simulation
    import gc
    
    # === Shuffle at data level (changes topology) ===
    if shuffle:
        # Create shuffled data copy
        data_use = data.copy()
        df_conn_shuffled = data['df_conn'].copy()
        
        # Shuffle post-synaptic targets
        # This changes WHO connects to WHOM
        post_col = data['columns']['post_col']
        post_ids = df_conn_shuffled[post_col].values.copy()
        
        np.random.seed(sim_idx)  # Reproducible shuffle
        np.random.shuffle(post_ids)
        
        df_conn_shuffled[post_col] = post_ids
        data_use['df_conn'] = df_conn_shuffled
    else:
        # Use original data
        data_use = data
    
    # Build network (with original or shuffled connectivity)
    columns = data_use['columns']
    net = build_network(
        data=data_use,
        pre_col=columns['pre_col'],
        post_col=columns['post_col'],
        weight_col=columns['weight_col'],
        nt_prob_cols=columns.get('nt_prob_cols', {}),
        params=params,
        syn_threshold=5,
        verbose=False
    )
    
    # Run simulation (100 Hz sugar activation, 1 trial)
    result = run_simulation(
        net_components=net,
        neu_exc=neu_exc,
        params={'r_poi': 100 * Hz},
        n_trials=1,
        verbose=False
    )
    
    # Check if MN9 activated (>0 Hz)
    df = result['df']
    mn9_activated = len(df[df['flywire_id'] == target_id]) > 0
    
    # Cleanup
    del net, result
    if shuffle:
        del data_use, df_conn_shuffled
    gc.collect()
    
    return (sim_idx, shuffle, mn9_activated, task_key)

In [ ]:
# Check existing results
if SHUFFLE_RESULTS_FILE.exists():
    print(f"✅ Loading cached results...")
    with open(SHUFFLE_RESULTS_FILE, 'rb') as f:
        shuffle_results = pickle.load(f)
    
    normal_act = shuffle_results['normal_activations']
    shuffled_act = shuffle_results['shuffled_activations']
    
    print(f"  Normal: {normal_act}/100")
    print(f"  Shuffled: {shuffled_act}/100")

else:
    print(f"Running {N_SIMULATIONS}×2 simulations...")
    print(f"  Method: Data-level shuffle (changes connection topology)")
    print(f"  Expected time: ~10 minutes (4 workers)")
    
    from flylif.utils.checkpoint import CheckpointManager
    ckpt = CheckpointManager(SHUFFLE_CHECKPOINT_DIR)
    
    # Create tasks
    tasks = []
    for i in range(N_SIMULATIONS):
        # Normal
        key_n = f'normal_{i}'
        if not ckpt.is_completed(key_n):
            tasks.append((i, False, DATA, DEFAULT_PARAMS, 
                         NEU_SUGAR_LEFT, NEU_MN9_RIGHT[0], key_n))
        
        # Shuffled
        key_s = f'shuffled_{i}'
        if not ckpt.is_completed(key_s):
            tasks.append((i, True, DATA, DEFAULT_PARAMS,
                         NEU_SUGAR_LEFT, NEU_MN9_RIGHT[0], key_s))
    
    print(f"  Remaining tasks: {len(tasks)}/200")
    
    if tasks:
        t0 = time()
        
        # Run with checkpoint
        from joblib import Parallel, delayed
        
        results = Parallel(n_jobs=4, verbose=5)(
            delayed(shuffled_control_worker)(*task) for task in tasks
        )
        
        # Save each result
        for sim_idx, shuffle, activated, task_key in results:
            ckpt.save(task_key, {'activated': activated})
        
        print(f"\n  ✅ Complete ({(time()-t0)/60:.1f} min)")
    
    # Load all results
    all_results = ckpt.load_all_completed()
    
    normal_act = sum(1 for k, v in all_results.items() 
                    if k.startswith('normal_') and v['activated'])
    shuffled_act = sum(1 for k, v in all_results.items()
                      if k.startswith('shuffled_') and v['activated'])
    
    # Save summary
    shuffle_results = {
        'normal_activations': normal_act,
        'shuffled_activations': shuffled_act,
        'n_simulations': N_SIMULATIONS,
    }
    
    with open(SHUFFLE_RESULTS_FILE, 'wb') as f:
        pickle.dump(shuffle_results, f)
    
    print(f"\n✅ Saved: {SHUFFLE_RESULTS_FILE}")

print(f"\n[Results]")
print(f"  Normal (true connectome):    {normal_act}/100 ({100*normal_act/100:.0f}%)")
print(f"  Shuffled (random topology):  {shuffled_act}/100 ({100*shuffled_act/100:.0f}%)")


## 11. Visualization - Neural and Motor Responses

In [ ]:
print("\n" + "=" * 70)
print("Generating Figures")
print("=" * 70)

# Create output directory
output_dir = Path('./results/exp1_sugar_activation')
fig_dir = output_dir / 'figures'
fig_dir.mkdir(parents=True, exist_ok=True)

### 11.1 Correlation plot (if original data available)

In [ ]:
if has_original:
    print("\nFigure 1: Correlation with original paper (100Hz)")
    
    fig1, ax1, r_val = plot_correlation(
        rate_orig, rate_ours, 
        n_trials_ours=10, 
        freq=100,
        # save_path=fig_dir / 'fig1_correlation_100Hz.png'
    )
    
    plt.show()
    print(f"  ✅ Saved: fig1_correlation_100Hz.png")
    print(f"  Correlation: r = {r_val:.4f}")

### 11.2 Response heatmap (Figure 1d style)

In [ ]:
print("\nFigure 2: Response heatmap (all frequencies)")

fig2, ax2, top_neurons = plot_response_heatmap(
    all_firing_rates, 
    freq_list=sorted(results_dict.keys()),
    cmap=sns.color_palette("Spectral_r", as_cmap=True),
    top_n=None,  # Auto-detect
    preset='auto',
    # save_path=fig_dir / 'fig2_response_heatmap.png'
)

plt.show()

print(f"  ✅ Saved: fig2_response_heatmap.png")
print(f"  Showing top {len(top_neurons)} neurons")

### 11.3 Proboscis motor neuron responses

In [ ]:
# MN9 bilateral (ipsi vs contra)
if NEU_MN9_RIGHT and NEU_MN9_LEFT:
    print("\nFigure 3: MN9 bilateral response curve")
    
    target_neurons = NEU_MN9_LEFT + NEU_MN9_RIGHT
    labels = ['MN9 Ipsilateral', 'MN9 Contralateral']
    
    fig3, ax3 = plot_frequency_response_curve(
        all_firing_rates,
        figsize=(6, 6),
        freq_list=sorted(results_dict.keys()),
        target_neurons=target_neurons,
        labels=labels,
        # save_path=fig_dir / 'fig3_mn9_bilateral_response.png'
    )
    
    plt.show()
    print(f"  ✅ Saved: fig3_mn9_bilateral_response.png")
else:
    print("\n⚠️  MN9 IDs not defined, skipping MN9 curve")

In [ ]:
# ========== 新增Cell: Section 11.4 ==========
# Additional motor neurons (MN6, 8, 11)
print("\nAdditional Figure: Other proboscis motor neurons")

# Define MN IDs (需要转换，这里用placeholder)
MN6_RIGHT = 720575940628826128
MN8_RIGHT = 720575940612888178
MN11_RIGHT = 720575940619048827

[MN6_ID, MN8_ID, MN11_ID] = convert_neuron_list_cached(
    old_list=[MN6_RIGHT,MN8_RIGHT,MN11_RIGHT],
    cache_file=cache_dir / 'mn6_v630_to_v783.pkl',
    new_ver=783,
    verbose=False,)

# 如果IDs可用，绘制
if all(x is not None for x in [MN6_ID, MN8_ID, MN11_ID]):
    fig, axes = plt.subplots(2, 2, figsize=(8, 6))
    axes = axes.flatten()
    
    mn_ids = [NEU_MN9_RIGHT[0], MN6_ID, MN8_ID, MN11_ID]
    mn_names = ['MN9', 'MN6', 'MN8', 'MN11']
    
    for ax, mn_id, name in zip(axes, mn_ids, mn_names):
        rates = [all_firing_rates[f].get(mn_id, 0) for f in sorted(all_firing_rates.keys())]
        ax.plot(sorted(all_firing_rates.keys()), rates, 'o-', linewidth=2, markersize=8)
        ax.set_xlabel('Sugar GRN Rate (Hz)', fontsize=11)
        ax.set_ylabel('Firing Rate (Hz)', fontsize=11)
        ax.set_title(f'{name} Response', fontsize=12, fontweight='bold')
        ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    # plt.savefig(fig_dir / 'fig_motor_neurons_panel.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    print(f"  ✅ Saved: fig_motor_neurons_panel.png")
else:
    print(f"  ⚠️  Other MN IDs not available, extracting from data...")

### 11.4 Shuffled Connectivity Validation

In [ ]:
# ========== Section 11.5: Shuffled Connectivity Validation ==========

print("\nFigure 5: Shuffled connectivity control")

# Load results (in case kernel restarted)
shuffle_file = Path('./results/exp1_full/shuffled_connectivity_results.pkl')

if shuffle_file.exists():
    with open(shuffle_file, 'rb') as f:
        shuffle_data = pickle.load(f)
    
    normal_act = shuffle_data['normal_activations']
    shuffled_act = shuffle_data['shuffled_activations']
    
    # Plot
    fig, ax = plt.subplots(figsize=(6, 6))
    
    bars = ax.bar(['True\nConnectome', 'Shuffled\nConnectome'],
                  [normal_act, shuffled_act],
                  color=['steelblue', 'coral'], 
                  edgecolor='black', alpha=0.8, width=0.6)
    
    ax.set_ylabel('MN9 Activations (out of 100 simulations)', fontsize=12)
    ax.set_title('Model Depends on True Connectivity\n(Sugar 100Hz → MN9)', 
                 fontsize=13, fontweight='bold')
    ax.set_ylim(0, 110)
    
    # Value labels
    for bar, val in zip(bars, [normal_act, shuffled_act]):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 3,
                f'{val}/100\n({100*val/100:.0f}%)', 
                ha='center', va='bottom', fontsize=11, fontweight='bold')
    
    # Statistical annotation
    from scipy.stats import fisher_exact
    contingency = [[normal_act, 100-normal_act],
                   [shuffled_act, 100-shuffled_act]]
    _, p_value = fisher_exact(contingency)
    
    y_max = max(normal_act, shuffled_act)
    ax.plot([0, 1], [y_max+15, y_max+15], 'k-', linewidth=1.5)
    ax.text(0.5, y_max+17, f'p < 0.001' if p_value < 0.001 else f'p = {p_value:.3f}',
            ha='center', fontsize=10, fontweight='bold')
    
    ax.grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    # plt.savefig(fig_dir / 'fig5_shuffled_control.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    print(f"  ✅ Saved: fig5_shuffled_control.png")
    print(f"  Statistical test: Fisher's exact, p = {p_value:.2e}")

else:
    print(f"  ⚠️  Shuffled results not found: {shuffle_file}")
    print(f"     Run Section 10.5 first")

### 11.5 Summary statistics

In [ ]:
print("\nFigure 4: Summary statistics")

fig4, axes4 = plot_summary_statistics(
    df_summary,
    save_path=fig_dir / 'fig4_summary_stats.png'
)

plt.show()
print(f"  ✅ Saved: fig4_summary_stats.png")

print(f"\n✅ All figures saved to: {fig_dir}")

## 12. Generate Top 200 Neurons

In [ ]:
print("\n" + "=" * 70)
print("Selecting Top 200 Responsive Neurons")
print("=" * 70)

# Sort by 200Hz firing rate (matching original paper)
if 200 in all_firing_rates:
    rates_200hz = pd.Series(all_firing_rates[200])
    top_200_neurons = rates_200hz.sort_values(ascending=False).head(200).index.tolist()
    
    print(f"\nTop 200 neurons (by 200Hz response):")
    print(f"  Count: {len(top_200_neurons)}")
    print(f"  Min rate: {rates_200hz[top_200_neurons[-1]]:.2f} Hz")
    print(f"  Max rate: {rates_200hz[top_200_neurons[0]]:.2f} Hz")
    
    # Save for Exp2/3
    np.save(output_dir / 'top_200_neurons.npy', top_200_neurons)
    
    # Save as text (readable)
    with open(output_dir / 'top_200_neurons.txt', 'w') as f:
        f.write("# Top 200 neurons by 200Hz firing rate\n")
        f.write("# Generated from Exp1 for use in Exp2/Exp3\n")
        f.write(f"# Date: {pd.Timestamp.now()}\n")
        f.write("#\n")
        for i, nid in enumerate(top_200_neurons, 1):
            f.write(f"{i:3d}. {nid} ({rates_200hz[nid]:.2f} Hz)\n")
    
    print(f"\n✅ Saved:")
    print(f"  - top_200_neurons.npy (for Exp2/3 code)")
    print(f"  - top_200_neurons.txt (human readable)")
    
    # Show examples
    print(f"\n  Top 10 neurons:")
    for i in range(10):
        nid = top_200_neurons[i]
        rate = rates_200hz[nid]
        print(f"    {i+1:2d}. {nid}: {rate:.1f} Hz")
    
else:
    print("\n❌ No 200Hz data, cannot select Top 200")
    print("   Check if Batch 4 completed successfully")

## 13. Save Complete Results

In [ ]:
print("\n" + "=" * 70)
print("Saving Complete Results Package")
print("=" * 70)

# Compile complete results package
results_package = {
    'results_dict': results_dict,
    'all_firing_rates': all_firing_rates,
    'df_summary': df_summary,
    'parameters': {
        'freq_list': sorted(results_dict.keys()),
        'n_trials': 10,
        'n_frequencies': len(results_dict),
        'neurons_activated': len(NEU_SUGAR_LEFT),
        'sugar_grn_ids': NEU_SUGAR_LEFT,
        'mn9_ids': {'left': NEU_MN9_LEFT, 'right': NEU_MN9_RIGHT},
    },
    'validation': {
        'correlation_100Hz': r if has_original else None,
        'top_neurons': top_200_neurons if 'top_200_neurons' in dir() else None,
        'original_comparison': df_compare.to_dict() if has_original else None,
    },
    'metadata': {
        'date': pd.Timestamp.now().isoformat(),
        'connectome_version': 783,
        'data_size_mb': DATA['df_conn'].memory_usage(deep=True).sum()/1e6,
    }
}

# Save complete package
with open(output_dir / 'exp1_complete_results.pkl', 'wb') as f:
    pickle.dump(results_package, f)

print(f"\n✅ Complete results saved:")
print(f"  {output_dir / 'exp1_complete_results.pkl'}")

# Save summary CSV
df_summary.to_csv(output_dir / 'summary.csv', index=False)
print(f"  {output_dir / 'summary.csv'}")

# Save firing rate matrix (neurons × frequencies)
if all_firing_rates:
    all_freqs = sorted(results_dict.keys())
    all_neurons = set()
    for rates_dict in all_firing_rates.values():
        all_neurons.update(rates_dict.keys())
    
    rate_matrix = pd.DataFrame(index=sorted(all_neurons), columns=all_freqs)
    for freq in all_freqs:
        for neuron_id, rate in all_firing_rates[freq].items():
            rate_matrix.loc[neuron_id, freq] = rate
    
    rate_matrix.fillna(0).to_csv(output_dir / 'firing_rate_matrix.csv')
    print(f"  {output_dir / 'firing_rate_matrix.csv'}")
    print(f"    ({rate_matrix.shape[0]} neurons × {rate_matrix.shape[1]} frequencies)")

## 14. Experiment Summary

In [ ]:
print("\n" + "=" * 70)
print("✅ Experiment 1 Complete!")
print("=" * 70)

print(f"\n[Key Results]")
print(f"  Frequencies tested: {len(results_dict)}")
if 'all_neurons' in dir():
    print(f"  Total active neurons: {len(all_neurons)}")
if has_original:
    print(f"  Correlation with paper: r = {r:.4f}")
if 'top_200_neurons' in dir():
    print(f"  Top 200 neurons saved: ✅")

print(f"\n[Outputs]")
print(f"  Results directory: {output_dir}/")
print(f"  - exp1_complete_results.pkl (full data)")
print(f"  - top_200_neurons.npy (for Exp2/3)")
print(f"  - summary.csv (table)")
print(f"  - firing_rate_matrix.csv (neurons × frequencies)")
print(f"  - figures/ (4 plots)")

print(f"\n[Next Steps]")
print(f"  1. Use top_200_neurons.npy in Exp2 (sufficiency test)")
print(f"  2. Use top_200_neurons.npy in Exp3 (necessity test)")
print(f"  3. Compare results with original paper figures")

print("\n" + "=" * 70)

---

## Appendix: Detailed Statistics

In [ ]:
# Detailed summary table
print("\nDetailed Summary (all frequencies):")
print("=" * 70)
print(df_summary.to_string(index=False))

# Firing rate statistics per frequency
print(f"\n\nFiring Rate Distribution per Frequency:")
print("=" * 70)

for freq in sorted(all_firing_rates.keys()):
    rates = list(all_firing_rates[freq].values())
    if rates:
        print(f"\n{freq:3d} Hz:")
        print(f"  Active neurons: {len(rates)}")
        print(f"  Mean rate: {np.mean(rates):.2f} Hz")
        print(f"  Median rate: {np.median(rates):.2f} Hz")
        print(f"  Max rate: {np.max(rates):.2f} Hz")
        print(f"  Std: {np.std(rates):.2f} Hz")

In [ ]:
# Check if specific neurons of interest are in top 200
if 'top_200_neurons' in dir() and NEU_MN9_RIGHT:
    print(f"\n\nNeurons of Interest in Top 200:")
    print("=" * 70)
    
    # MN9
    for mn9_id, side in [(NEU_MN9_LEFT[0], 'Left'), (NEU_MN9_RIGHT[0], 'Right')]:
        if mn9_id in top_200_neurons:
            rank = top_200_neurons.index(mn9_id) + 1
            rate = rates_200hz[mn9_id]
            print(f"  MN9 {side}: Rank {rank}/200 ({rate:.1f} Hz)")
        else:
            print(f"  MN9 {side}: Not in top 200")